# J2-04 · Robustesse : retries, timeouts, gestionnaires d'erreur, sous-graphes

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 2 · créneau 14h30–16h00 · ⏱️ ~1h30 (dont l'exercice 2.2, 40 min)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Réessayer une étape qui échoue (`RetryPolicy`) · 2. Borner une étape trop lente (`TimeoutPolicy`, nœuds **async**) · 3. Se replier proprement quand tout a échoué (`error_handler` + `NodeError`) · 4. Éviter les pièges (`ToolNode` qui avale les erreurs, handler sans `goto`) · 5. Composer avec des **sous-graphes** |
| 🧱 **Fil rouge** | L'outil `ping_serveur` interroge une supervision **instable** : HelpDesk-Ops doit tenir le coup |
| ✅ **Prérequis** | J2-01 à J2-03 |
| 📦 **Versions** | `langgraph 1.2` (timeouts et gestionnaires d'erreur par nœud : nouveautés 1.2) |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Ce qui casse dans un agent

### [Théorie] Trois familles de pannes, trois réponses

| Panne | Exemple HelpDesk-Ops | Réponse LangGraph 1.2 |
|---|---|---|
| **transitoire** | la supervision répond « connexion refusée » une fois sur trois | `RetryPolicy` : réessayer, avec attente croissante |
| **lenteur** | un LLM local sur CPU met 2 minutes à répondre | `TimeoutPolicy` : abandonner au-delà d'un délai |
| **persistante** | la supervision est arrêtée pour maintenance | `error_handler` : se replier (réponse dégradée, escalade) |

*Analogie : le standard téléphonique.* On rappelle un numéro occupé (retry), on raccroche au bout de 10 sonneries (timeout), et si personne ne répond on laisse un message à l'accueil (repli).

> 🛡️ **OWASP ASI08 (Cascading Failures) :** sans ces protections, une panne d'outil devient une boucle d'appels LLM, puis une indisponibilité de tout le service.

## 3. `RetryPolicy` : réessayer ce qui est transitoire

In [ ]:
import asyncio
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, TimeoutPolicy

from helpdesk_ops.reseau import RESEAU


class Diagnostic(TypedDict, total=False):
    hote: str
    resultat: str
    journal: Annotated[list[str], operator.add]


async def pinger(state: Diagnostic) -> dict:
    return {"resultat": await RESEAU.ping(state["hote"]), "journal": [f"ping réussi à l'appel n°{RESEAU.appels}"]}


def graphe_ping(**options_noeud):
    b = StateGraph(Diagnostic)
    b.add_node("pinger", pinger, **options_noeud)
    b.add_edge(START, "pinger")
    b.add_edge("pinger", END)
    return b.compile()


RESEAU.configurer(echecs_avant_succes=2)   # 2 échecs, puis succès
retry = RetryPolicy(max_attempts=3, initial_interval=0.2, backoff_factor=2.0, retry_on=ConnectionError)
debut = time.perf_counter()
print(await graphe_ping(retry_policy=retry).ainvoke({"hote": "srv-mail-01"}))
print(f"appels : {RESEAU.appels} en {time.perf_counter() - debut:.1f} s (attentes 0,2 s puis 0,4 s, ± gigue)")

**👀 Ce que vous devez observer :** 3 appels, et le graphe réussit. Les attentes doublent (`backoff_factor=2`), avec une gigue aléatoire (`jitter=True` par défaut) pour éviter que 50 clients ne réessaient à la même milliseconde.

> 💡 **Le « Tips » du Formateur :** `retry_on` est **crucial**. On réessaie une `ConnectionError`, jamais une `ValueError` (un argument invalide restera invalide) ni une erreur d'autorisation. Par défaut (`default_retry_on`, lu dans le code de LangGraph 1.2) : `ConnectionError` et erreurs HTTP 5xx sont réessayées ; `ValueError`, `TypeError`, `KeyError`, `RuntimeError` et `OSError` ne le sont pas, **y compris `TimeoutError`** (sous-classe d'`OSError`) ; toute autre exception l'est. Soyez explicite.

> ⚠️ Un retry **rejoue le nœud entier**. Si le nœud a déjà écrit quelque chose avant de planter, c'est un doublon : même règle d'idempotence qu'en J2-02.

## 4. `TimeoutPolicy` : borner ce qui est lent, en **async** uniquement

### [Théorie] Pourquoi async ?

Python ne sait pas interrompre proprement une fonction synchrone en cours. LangGraph refuse donc, **dès la compilation**, un timeout sur un nœud synchrone. Un nœud `async def` peut, lui, être annulé.

| Paramètre | Rôle |
|---|---|
| `run_timeout` | durée maximale d'une tentative (horloge murale) |
| `idle_timeout` | durée maximale **sans progrès** (utile en streaming : un modèle qui produit des tokens n'est pas bloqué) |

In [ ]:
def pinger_sync(state: Diagnostic) -> dict:
    return {"resultat": "ok"}


b = StateGraph(Diagnostic)
b.add_node("pinger", pinger_sync, timeout=TimeoutPolicy(run_timeout=1))
b.add_edge(START, "pinger")
try:
    b.compile()
except ValueError as erreur:
    print("❌", str(erreur)[:120], "…")

In [ ]:
from langgraph.errors import NodeTimeoutError

RESEAU.configurer(latence_s=2.0)   # la supervision met 2 s à répondre
try:
    await graphe_ping(timeout=TimeoutPolicy(run_timeout=0.5)).ainvoke({"hote": "srv-mail-01"})
except NodeTimeoutError as erreur:
    print("⏱️", erreur)

**👀 Ce que vous devez observer :** `NodeTimeoutError` au bout de 0,5 s, au lieu d'attendre 2 s. Combiné à une `RetryPolicy` qui inclut `NodeTimeoutError` dans `retry_on`, le timeout déclenche un nouvel essai.

> 💡 **Le « Tips » du Formateur :** réglez les timeouts **sur le matériel réel**. Sur le poste de préparation (CPU), un appel au modèle 8B prend 30 à 60 s : un timeout de 30 s y échouerait toujours. En salle (GPU), 30 s est confortable. D'où un paramètre, pas une constante.

## 5. `error_handler` : le repli quand tout a échoué

### [Théorie] Un nœud de secours, déclaré sur le nœud qui peut échouer

Quand les tentatives sont épuisées (ou sans `RetryPolicy`), LangGraph appelle le **gestionnaire d'erreur** du nœud au lieu de faire échouer tout le graphe. Il reçoit l'état et, s'il déclare un paramètre typé **`NodeError`**, l'erreur elle-même (`error.node`, `error.error`).

> ⚠️ **Piège vérifié en préparant ce cours :** après un gestionnaire d'erreur, les **arêtes normales du nœud ne sont pas suivies**. Un gestionnaire qui renvoie un simple `dict` arrête le graphe. Pour continuer, renvoyez `Command(update=…, goto="nœud_suivant")`.

In [ ]:
from langgraph.errors import NodeError


def repli(state: Diagnostic, error: NodeError) -> Command:
    message = f"supervision indisponible ({type(error.error).__name__} dans « {error.node} ») : escalade N2"
    return Command(update={"resultat": message, "journal": ["repli activé"]}, goto="conclure")


def conclure(state: Diagnostic) -> dict:
    return {"journal": [f"conclusion : {state['resultat']}"]}


b = StateGraph(Diagnostic)
b.add_node("pinger", pinger, retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=ConnectionError),
           error_handler=repli)
b.add_node("conclure", conclure)
b.add_edge(START, "pinger")
b.add_edge("pinger", "conclure")
b.add_edge("conclure", END)
robuste = b.compile()

for scenario, reglages in {"panne passagère": {"echecs_avant_succes": 1}, "panne durable": {"echecs_avant_succes": 99}}.items():
    RESEAU.configurer(**reglages)
    etat = await robuste.ainvoke({"hote": "srv-erp-02"})
    print(f"{scenario:<16} | {RESEAU.appels} appel(s) | {etat['journal']}")

**👀 Ce que vous devez observer :** la panne passagère est absorbée par le retry (2 appels) ; la panne durable épuise les 3 tentatives puis passe par le repli, et le graphe **continue** vers `conclure` grâce au `goto`.

### [Code] `set_node_defaults` : une politique pour tous les nœuds

In [ ]:
b = StateGraph(Diagnostic)
b.set_node_defaults(retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.1, retry_on=ConnectionError))
b.add_node("pinger", pinger)                                 # hérite de la politique par défaut
b.add_node("conclure", conclure, retry_policy=RetryPolicy(max_attempts=1))   # surcharge explicite
b.add_edge(START, "pinger")
b.add_edge("pinger", "conclure")
b.add_edge("conclure", END)
RESEAU.configurer(echecs_avant_succes=1)
etat = await b.compile().ainvoke({"hote": "srv-vpn-01", "resultat": ""})
print(f"{RESEAU.appels} appels → {etat['resultat']}")

## 6. Le piège du `ToolNode` : quand les erreurs sont avalées

### [Théorie] Qui gère l'erreur d'un outil ?

Dans un agent, les outils s'exécutent dans un `ToolNode`. Son paramètre `handle_tool_errors` décide du sort des exceptions :

| `handle_tool_errors` | Comportement | Conséquence pour `RetryPolicy` |
|---|---|---|
| défaut | erreurs d'**arguments** renvoyées au modèle ; les autres **remontent** | ✅ le retry du nœud fonctionne |
| `True` | **toute** exception devient un `ToolMessage` d'erreur pour le modèle | ❌ le retry ne se déclenche jamais |

Avec `True`, c'est le **LLM** qui décide de réessayer… ou d'inventer un résultat. Pour une panne réseau, on préfère un retry déterministe.

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

from helpdesk_ops.reseau import ping_serveur

from langgraph.graph import MessagesState


def graphe_outils(**options_toolnode):
    """Un ToolNode s'exécute DANS un graphe (LangGraph 1.2 refuse de l'appeler seul) : graphe d'un seul nœud."""
    b = StateGraph(MessagesState)
    b.add_node("tools", ToolNode([ping_serveur], **options_toolnode))
    b.add_edge(START, "tools")
    b.add_edge("tools", END)
    return b.compile()


appel = AIMessage("", tool_calls=[{"name": "ping_serveur", "args": {"hote": "srv-erp-02"}, "id": "appel-1"}])

RESEAU.configurer(echecs_avant_succes=99)
sortie = await graphe_outils(handle_tool_errors=True).ainvoke({"messages": [appel]})
print("handle_tool_errors=True →", sortie["messages"][-1].status, ":", sortie["messages"][-1].content[:80])

try:
    await graphe_outils().ainvoke({"messages": [appel]})
except ConnectionError as erreur:
    print("défaut                  → l'exception remonte :", erreur)

**👀 Ce que vous devez observer :** avec `True`, aucune exception : l'erreur part au modèle comme un résultat. Avec le défaut, l'exception remonte, et la `RetryPolicy` du nœud `tools` peut agir.

Et si les retries échouent quand même ? Le gestionnaire d'erreur du nœud `tools` doit renvoyer **un `ToolMessage` par appel d'outil en attente** (sinon l'historique est invalide pour le modèle), puis `goto="agent"`. C'est exactement ce que fait le jalon J2 (`helpdesk_ops/jalons/j2.py`, fonction `outils_en_echec`).

## 7. Sous-graphes : composer sans tout mélanger

### [Théorie] Un graphe compilé est un nœud comme un autre

Un graphe compilé peut être ajouté comme **nœud** d'un autre graphe. Il partage les clés d'état communes (ici `messages`) et garde ses clés privées. Intérêts : réutiliser (le diagnostic sert à plusieurs agents), tester séparément, donner une politique de robustesse à tout un bloc.

*Analogie :* un service de l'entreprise. Le help-desk lui transmet un dossier, le service fait son travail interne, puis rend le dossier complété.

In [ ]:
class EtatDiagnostic(MessagesState):
    hote: str          # clé privée du sous-graphe
    resultat: str


def extraire_hote(state: EtatDiagnostic) -> dict:
    texte = state["messages"][-1].content.lower()
    return {"hote": next((mot.strip(" ?.,") for mot in texte.split() if mot.startswith("srv-")), "inconnu")}


async def diagnostiquer(state: EtatDiagnostic) -> dict:
    return {"resultat": await RESEAU.ping(state["hote"])}


def rapporter(state: EtatDiagnostic) -> dict:
    return {"messages": [AIMessage(f"Diagnostic réseau : {state['resultat']}")]}


sb = StateGraph(EtatDiagnostic)
sb.add_node("extraire_hote", extraire_hote)
sb.add_node("diagnostiquer", diagnostiquer, retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=ConnectionError))
sb.add_node("rapporter", rapporter)
sb.add_edge(START, "extraire_hote")
sb.add_edge("extraire_hote", "diagnostiquer")
sb.add_edge("diagnostiquer", "rapporter")
sb.add_edge("rapporter", END)
sous_graphe_diagnostic = sb.compile()

# Le graphe parent ne connaît que `messages` : le sous-graphe est une boîte noire réutilisable
pb = StateGraph(MessagesState)
pb.add_node("diagnostic", sous_graphe_diagnostic)
pb.add_edge(START, "diagnostic")
pb.add_edge("diagnostic", END)
parent = pb.compile()

RESEAU.configurer(echecs_avant_succes=1)
with propagate_attributes(session_id="j2-04-sous-graphe", tags=["jour-2", "robustesse"]):
    etat = await parent.ainvoke({"messages": [("user", "srv-erp-02 répond-il ?")]},
                                config={"callbacks": [langfuse_handler], "run_name": "sous-graphe"})
print(etat["messages"][-1].content, "| clés visibles du parent :", sorted(etat))
langfuse.flush()

**👀 Ce que vous devez observer :** le parent ne voit que `messages` (les clés `hote` et `resultat` restent internes) ; dans Langfuse, la trace `sous-graphe` montre le nœud `diagnostic` **avec ses étapes internes imbriquées**, dont la tentative en échec.

## 8. 🧪 Exercice 2.2 (40 min)

➡️ `exercices/enonces/Ex2_2_reseau_instable.ipynb` : rendre l'agent support tolérant aux pannes de la supervision, et le **prouver** dans Langfuse.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Réessayer une panne transitoire | `add_node(..., retry_policy=RetryPolicy(max_attempts, initial_interval, retry_on=…))` |
| Borner une lenteur | `timeout=TimeoutPolicy(run_timeout=…, idle_timeout=…)`, nœud `async def` |
| Se replier | `error_handler=f(state, error: NodeError) -> Command(update=…, goto=…)` |
| Une politique commune | `StateGraph.set_node_defaults(...)` |
| Garder la main sur les erreurs d'outils | `ToolNode` par défaut (pas `handle_tool_errors=True`) + retry sur le nœud `tools` |
| Composer | un graphe compilé comme nœud (sous-graphe) |

### ✅ Quiz express

1. Pourquoi un timeout est-il refusé sur un nœud synchrone ?
2. Quelle exception ne faut-il jamais mettre dans `retry_on` ? Donnez un exemple.
3. Un gestionnaire d'erreur renvoie `{"resultat": "repli"}`. Que se passe-t-il ensuite ?
4. Pourquoi `handle_tool_errors=True` désactive-t-il en pratique la `RetryPolicy` du nœud `tools` ?

<details>
<summary>Réponses</summary>

1. Python ne peut pas annuler proprement une fonction synchrone en cours ; une coroutine `async`, si.
2. Une erreur qui se reproduira à l'identique : `ValueError` (argument invalide), erreur d'authentification, `KeyError` de programmation.
3. L'état est mis à jour, mais les arêtes du nœud ne sont pas suivies : le graphe s'arrête. Il faut `Command(update=…, goto=…)`.
4. Le `ToolNode` transforme l'exception en `ToolMessage` : aucune exception ne remonte, le nœud « réussit », le retry ne se déclenche pas.
</details>